
# VadCLIP + Nhất Quán Theo Dịch Chuyển — Bản Kaggle (lần chạy đối chứng λ = 0)

Bản Kaggle của `train_shift_consistency_v3_colab.ipynb`, thu gọn về **đúng một việc**:
chạy lần đối chứng với trọng số của số hạng thứ tư bằng 0, rồi đo nó bằng cùng thước đo
mà các lần chạy có ràng buộc đã dùng.

Vì sao lần chạy này đáng một phiên GPU riêng: mọi kết luận trong hướng nghiên cứu này đều
là **hiệu** giữa một cấu hình và đối chứng cùng hạt giống. Sàn nhiễu đo được ở vòng 2 là
2,56 điểm AUC giữa hai lần đối chứng chỉ khác hạt giống — lớn hơn mọi hiệu ứng mà λ tạo
ra. Với đúng hai mẫu, con số đó mới là một quan sát chứ chưa phải một thống kê. Mẫu thứ
ba biến nó thành ước lượng có biên độ, và đó là thứ quyết định bảng kết quả cuối cùng có
đọc được hay không.

## Ba thay đổi trong mã nguồn so với bản Colab

| Cờ mới | Mặc định | Việc của nó |
|---|---|---|
| `--deterministic` | `false` | Ghim cudnn về kernel tất định, tắt benchmark, đặt `CUBLAS_WORKSPACE_CONFIG`. Bản sao của cờ cùng tên trong `baseline/src/ucf_train.py`. Chạy lại cùng lệnh trên cùng GPU ra cùng trọng số |
| `--skip-shifted-view` | `false` | Chỉ đưa view đầy đủ qua mô hình, tức **giảm một nửa lô thị giác**. Chỉ hợp lệ khi λ đồng nhất bằng 0, và script từ chối chạy nếu không phải vậy |
| — | — | `setup_seed` nhận thêm tham số `deterministic` |

Cả ba đều mặc định giữ nguyên hành vi cũ, nên mọi lần chạy vòng 1, 2, 3 vẫn tái lập được
nguyên vẹn.

**Vì sao có `--skip-shifted-view`.** Lần đối chứng λ = 0 vẫn cắt view dịch, vẫn ghép nó
vào chiều lô, vẫn cho cả hai đi qua mô hình, rồi nhân số hạng nhất quán với đúng số 0.
Một nửa thời gian GPU của lần chạy đó không mua được gì. Bỏ view dịch là hợp lệ vì hai
điều đã được kiểm chứng: `CLIPVAD` độc lập theo lô — `tests/test_two_view_batching.py`
kiểm tra đúng điều đó trên mô hình thật — và một số hạng nhân với 0 không đóng góp
gradient nào.

**Nhưng mặc định vẫn là tắt.** Lý do khoa học chứ không phải kỹ thuật: đối chứng tồn tại
để khác lần chạy có ràng buộc **đúng một biến**. Cho nó đi một đường code khác là thêm
biến thứ hai. Chỉ bật cờ này khi giờ GPU là ràng buộc đang siết — mục 7 đo cả tốc độ lẫn
tính tương đương ngay trên dữ liệu thật để bạn quyết định bằng số liệu.

## Chuẩn bị trước khi mở notebook

### Code — chọn một trong hai

**Cách A (khuyến nghị): clone từ GitHub.** Không phải tạo dataset nào cho code. Mục 1 tự
clone `vngclinh/Finetune-VadCLIP` vào `/kaggle/working/repo`, và chạy lại cell đó là
tự cập nhật về commit mới nhất.

> ⚠️ **Bắt buộc push trước.** Clone chỉ lấy được thứ đã nằm trên GitHub. Toàn bộ code
> shift-consistency (`ucf_train_augment.py`, `ucf_option_augment.py`,
> `utils/dataset_augment.py`, `utils/layers.py` bản đã vá, `tests/`,
> `list/make_gt_ucf_relative.py`) phải được commit và push lên branch `main` trước.
> Quên bước này thì mục 4 sẽ báo "bản cũ" và dừng — nó kiểm tra danh sách tham số thật
> chứ không chỉ kiểm tra file có tồn tại.

**Cách B: upload thành Dataset.** Dùng khi không muốn push code lên GitHub.

```
src/                      ← copy nguyên thư mục VadCLIP/src/ (kèm clip/, utils/, tests/)
list/                     ← copy nguyên thư mục VadCLIP/list/
model_ucf.pth             ← tuỳ chọn, mốc đối chiếu ở mục 9
ViT-B-16.pt               ← tuỳ chọn, xem mục 3
```

Mục 1 tự chọn: có dataset code trong `/kaggle/input` thì dùng dataset, không có thì clone.
Ép một chiều bằng `CODE_SOURCE = 'github'` hoặc `'dataset'`.

### Feature — bắt buộc là Dataset

Dùng dataset công khai **`beosngu/ucf-crime-vadclip-features`** (14 GB): panel bên phải →
**Add Input** → **Datasets** → dán đường dẫn đó → **Add**.

Cấu trúc của nó là các thư mục lớp `Abuse/`, `Arson/`, …, `Normal_Videos_event/` chứa
file `.npy` ngay ở gốc — đúng thứ notebook cần.

Notebook **tự dò** trong `/kaggle/input`, nên đặt slug tên gì cũng được.

### `model_ucf.pth` và `ViT-B-16.pt`

Hai file này không có trên GitHub (quá nặng). Muốn có thì bỏ vào một Dataset riêng bất kỳ
— mục 1 dò `model_ucf.pth` và mục 3 dò `ViT-B-16.pt` ở mọi nơi trong `/kaggle/input`.
Thiếu cũng chạy được: mục 9 bỏ qua mốc đối chiếu, và CLIP tự tải 335 MB mỗi phiên.

## Cài đặt Session

| Mục | Giá trị | Vì sao |
|---|---|---|
| Accelerator | **GPU** T4 hoặc P100 | Bắt buộc |
| Internet | **On** | `pip install ftfy`, và tải trọng số CLIP nếu không nạp sẵn |
| Persistence | Files + Variables | Giữ `/kaggle/working` giữa các phiên |

## ⚠️ Hai giới hạn cứng của Kaggle

Phiên GPU tối đa **12 giờ**, quota **30 giờ/tuần**. `ucf_train_augment.py` **không có
cơ chế resume** — phiên đứt là mất trắng lần chạy. Mục 7 ước lượng tổng thời gian từ 20
bước thật; đọc con số đó trước khi bấm chạy mục 8.

`/kaggle/working` giới hạn **20 GB**. Mỗi file trọng số khoảng 350 MB và một lần chạy 10
epoch ghi ra 10 checkpoint theo epoch. Notebook này đẩy toàn bộ file trung gian sang
`/kaggle/temp` (không tính vào Output), chỉ giữ lại trọng số cuối.

Cách an toàn nhất: **Save Version → Save & Run All (Commit)**. Nó chạy nền, không cần giữ
trình duyệt mở.


## 1. Cấu Hình

Tự dò dataset trong `/kaggle/input`. Dò sai thì điền tay vào ba biến `*_OVERRIDE`.

Code phải nằm ở nơi ghi được (`ucf_train_augment.py` ghi `model/` theo thư mục hiện
hành), mà `/kaggle/input` là chỉ đọc — nên cell này copy `src/` và `list/` sang
`/kaggle/working`. Thư mục `list/` được copy chứ không đọc thẳng vì mục 5.1 có thể phải
sinh lại ba file ground truth vào đó.

In [ ]:

from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK       = Path('/kaggle/working')
TEMP       = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')

# --- Code lấy từ đâu ------------------------------------------------------------------
# 'auto'    : có dataset code trong /kaggle/input thì dùng, không có thì clone GitHub
# 'github'  : luôn clone, kể cả khi có dataset
# 'dataset' : luôn dùng dataset, không clone
CODE_SOURCE   = 'auto'
# Chi de IN RA trong thong bao goi y. KHONG dieu khien viec do duong dan --
# cai do la FEATURE_OVERRIDE ben duoi.
FEATURE_DATASET_HINT = 'beosngu/ucf-crime-vadclip-features'   # 14 GB, cong khai
GITHUB_REPO   = 'https://github.com/vngclinh/Finetune-VadCLIP.git'
GITHUB_BRANCH = 'main'

# Điền tay nếu tự dò sai. Ví dụ: Path('/kaggle/input/vadclip-shift')
CODE_OVERRIDE    = None
FEATURE_OVERRIDE = None
CKPT_OVERRIDE    = None


def walk_dirs(root, maxdepth=8):
    """Duyet thu muc theo be rong, CO di xuyen symlink.

    Khong dung rglob: `**` cua pathlib goi is_dir(follow_symlinks=False), tuc la no
    co y bo qua thu muc symlink. Kaggle mount dataset bang symlink, nen rglob khong
    bao gio nhin thay gi ben trong /kaggle/input. iterdir() + is_dir() thi di xuyen
    binh thuong.

    resolve() dung de khong di vong quanh mai neu co symlink tro nguoc lai.
    """
    if not root.exists():
        return
    seen = set()
    queue = [(root, 0)]
    while queue:
        directory, depth = queue.pop(0)
        try:
            key = directory.resolve()
        except OSError:
            key = directory
        if key in seen:
            continue
        seen.add(key)
        yield directory
        if depth >= maxdepth:
            continue
        try:
            queue.extend((child, depth + 1)
                         for child in sorted(directory.iterdir()) if child.is_dir())
        except (PermissionError, OSError):
            pass


def find_in_input(*markers, maxdepth=8):
    """Thu muc con cua /kaggle/input chua du cac duong dan danh dau."""
    for directory in walk_dirs(INPUT_ROOT, maxdepth):
        if all((directory / m).exists() for m in markers):
            return directory
    return None


def clone_repo():
    '''Clone repo vào /kaggle/working và trả về thư mục chứa src/ và list/.

    Repo có cấu trúc VadCLIP/src và VadCLIP/list, nên CODE_ROOT là thư mục VadCLIP.
    Đã clone rồi thì fetch + reset về đúng branch, để chạy lại cell là lấy bản mới nhất
    chứ không phải giữ bản cũ.
    '''
    clone_dir = WORK / 'repo'
    if (clone_dir / '.git').exists():
        print('Đã có repo, cập nhật về bản mới nhất ...')
        subprocess.run(['git', '-C', str(clone_dir), 'fetch', '--depth', '1',
                        'origin', GITHUB_BRANCH], check=True)
        subprocess.run(['git', '-C', str(clone_dir), 'reset', '--hard',
                        f'origin/{GITHUB_BRANCH}'], check=True)
    else:
        print('Clone', GITHUB_REPO, '...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH,
                        GITHUB_REPO, str(clone_dir)], check=True)
    sha = subprocess.run(['git', '-C', str(clone_dir), 'rev-parse', '--short', 'HEAD'],
                         capture_output=True, text=True).stdout.strip()
    print('Commit:', sha)
    return clone_dir / 'VadCLIP'


CODE_FROM_GITHUB = False
CODE_ROOT = CODE_OVERRIDE
if CODE_ROOT is None and CODE_SOURCE != 'github':
    CODE_ROOT = find_in_input('src/ucf_train_augment.py',
                              'list/ucf_CLIP_rgbtest_relative.csv')
    if CODE_ROOT is None:
        CODE_ROOT = find_in_input('src/ucf_train_augment.py')  # thiếu list -> preflight báo
if CODE_ROOT is None and CODE_SOURCE in ('auto', 'github'):
    CODE_ROOT = clone_repo()
    CODE_FROM_GITHUB = True

def find_feature_root():
    """Thu muc chua cac thu muc lop.

    Ba cach, thu lan luot. Tat ca deu chay tren walk_dirs nen di xuyen symlink va
    khong phu thuoc dataset long may tang.
    """
    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir() and (directory / 'Vandalism').is_dir():
            return directory, 'thay Abuse + Vandalism'

    for directory in walk_dirs(INPUT_ROOT):
        if (directory / 'Abuse').is_dir():
            return directory, 'chi thay Abuse'

    # Moi ten lop deu bi doi: tim thu muc co tu 10 thu muc con chua .npy.
    for directory in walk_dirs(INPUT_ROOT):
        try:
            children = [d for d in directory.iterdir() if d.is_dir()]
        except (PermissionError, OSError):
            continue
        with_npy = [d for d in children if next(d.glob('*.npy'), None) is not None]
        if len(with_npy) >= 10:
            return directory, f'{len(with_npy)} thu muc con co file .npy'
    return None, None


def print_input_tree(levels=6):
    '''In cay /kaggle/input de nhin ra ngay cau truc that cua dataset.'''
    print()
    print('=' * 78)
    print('/kaggle/input dang co gi:')
    if not INPUT_ROOT.exists():
        print('   (khong ton tai -- chua Add Input dataset nao)')
        return
    entries = sorted(INPUT_ROOT.iterdir())
    if not entries:
        print('   (rong -- chua Add Input dataset nao)')
        return
    def walk(path, depth):
        if depth > levels:
            return
        try:
            children = sorted(path.iterdir())
        except (PermissionError, OSError):
            return
        for child in children[:15]:
            npy = len(list(child.glob('*.npy'))) if child.is_dir() else 0
            mark = f'  <- {npy} file .npy' if npy else ''
            print('   ' + '   ' * depth + child.name + ('/' if child.is_dir() else '') + mark)
            if child.is_dir() and not npy:
                walk(child, depth + 1)
        if len(children) > 15:
            print('   ' + '   ' * depth + f'... con {len(children) - 15} muc nua')
    walk(INPUT_ROOT, 0)
    print('=' * 78)


if FEATURE_OVERRIDE is not None:
    FEATURE_ROOT, how = FEATURE_OVERRIDE, 'FEATURE_OVERRIDE dat tay'
else:
    FEATURE_ROOT, how = find_feature_root()

if FEATURE_ROOT is None:
    print('CHUA DO RA DATASET FEATURE.')
    print('Panel ben phai -> Add Input -> Datasets -> dan:', FEATURE_DATASET_HINT)
    print('Da add roi ma van bao the nay thi thu Run -> Restart & Clear Cell Outputs,')
    print('vi Kaggle doi khi can restart session moi thay input moi.')
    print_input_tree()
    print('Nhin cay tren, tim thu muc CHUA cac thu muc lop, roi dat o muc 1:')
    print("    FEATURE_OVERRIDE = Path('/kaggle/input/<ten-dataset>')")
else:
    print('Feature do ra bang:', how)
    print('Do sau tinh tu /kaggle/input:',
          len(FEATURE_ROOT.relative_to(INPUT_ROOT).parts), 'tang')
CKPT_ROOT = CKPT_OVERRIDE or find_in_input('model_ucf.pth') or CODE_ROOT

if CODE_ROOT is None:
    print('Có gì trong /kaggle/input:')
    for p in sorted(INPUT_ROOT.glob('*/*'))[:40]:
        print('  ', p)
    raise FileNotFoundError(
        'Không tìm thấy code. Đặt CODE_SOURCE = "github" để clone, hoặc điền CODE_OVERRIDE.')

# --- Code sang nơi ghi được -----------------------------------------------------------
PROJECT  = WORK / 'vadclip'
SRC_DIR  = PROJECT / 'src'
LIST_DIR = PROJECT / 'list'
if not SRC_DIR.exists():
    print('Copy code sang thư mục ghi được ...')
    shutil.copytree(CODE_ROOT / 'src', SRC_DIR)
if not LIST_DIR.exists() and (CODE_ROOT / 'list').exists():
    shutil.copytree(CODE_ROOT / 'list', LIST_DIR)
# __pycache__ đi theo từ dataset sẽ che mất file .py mới. Xoá cho chắc.
for cache in SRC_DIR.rglob('__pycache__'):
    shutil.rmtree(cache, ignore_errors=True)

PAPER_MODEL = (CKPT_ROOT / 'model_ucf.pth') if CKPT_ROOT else None

# --- Nơi ghi kết quả ------------------------------------------------------------------
# Giữ lại (vào Output): trọng số cuối, log, CSV, bảng độ nhạy.
# Vứt đi (sang /kaggle/temp): checkpoint theo epoch, model_cur, checkpoint chọn-theo-AUC.
RESULT_DIR = WORK / 'results'
LOG_DIR    = RESULT_DIR / 'logs'
MODEL_DIR  = WORK / 'models'
SCRATCH    = TEMP / 'vadclip_scratch'
for directory in (RESULT_DIR, LOG_DIR, MODEL_DIR, SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)

METRICS_CSV = str(RESULT_DIR / 'shift_kaggle_metrics.csv')

TRAIN_LIST = str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')
TEST_LIST  = str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv')
GT_ARGS = [
    '--gt-path',         str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path',   str(LIST_DIR / 'gt_label_ucf.npy'),
]

# ================= GIAO THỨC =================
# 'round1'  : tái lập lần chạy đã cho 88,13. Chọn checkpoint theo AUC tốt nhất TRÊN TẬP
#             TEST (luật upstream), chấm điểm ~12 lần mỗi epoch, hạt giống 234.
# 'round23' : giao thức sạch của vòng 2/3. Giữ trọng số CUỐI, chấm một lần mỗi epoch.
#             Dùng khi đo sàn nhiễu, không dùng khi tái lập một con số cũ.
#
# Cùng seed 234, cùng lambda = 0, chỉ khác luật chọn checkpoint:
#     round1  -> 88,13     round23 -> 87,84   (v2_ctrl_s234 đo được)
# Chênh 0,29 điểm đó là của LUẬT CHỌN, không phải của mô hình.
PROTOCOL = 'round1'
# =============================================

MAX_EPOCH    = 10
LR           = '2e-5'
BATCH_SIZE   = 64
NUM_WORKERS  = 4          # vòng 1 và vòng 2 đều dùng 4
USE_PRETRAINED = False    # train từ đầu, không nạp model_ucf.pth

if PROTOCOL == 'round1':
    SEEDS         = [234]     # đúng hạt giống của lần chạy 88,13
    SELECT_METRIC = 'auc'     # giữ trọng số tốt nhất trên tập test — luật của vòng 1
    EVAL_STEPS    = 1280      # ~12 lần chấm mỗi epoch; select-metric auc cần nó
    RUN_PREFIX    = 'repro_ctrl'
elif PROTOCOL == 'round23':
    SEEDS         = [777]     # hạt giống thứ ba; 234 và 1234 đã chạy trên Colab
    SELECT_METRIC = 'none'    # giữ trọng số cuối
    EVAL_STEPS    = 0         # chấm một lần mỗi epoch
    RUN_PREFIX    = 'kg_ctrl'
else:
    raise ValueError(f"PROTOCOL phải là 'round1' hoặc 'round23', không phải {PROTOCOL!r}")

# Hai cờ mới. Đọc phần đầu notebook trước khi đổi.
DETERMINISTIC     = False   # False = giống hệt mọi lần chạy shift-consistency trước đây
SKIP_SHIFTED_VIEW = False   # True = nhanh gần gấp đôi, nhưng đối chứng đi đường code khác
# ===============================================================

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)
PY = [sys.executable, '-u']

print('Nguồn code :', 'GitHub (' + GITHUB_BRANCH + ')' if CODE_FROM_GITHUB else 'Dataset ' + str(CODE_ROOT))
print('Code       :', SRC_DIR)
print('List       :', LIST_DIR)
print('Feature    :', FEATURE_ROOT)
print('Checkpoint :', PAPER_MODEL)
print('Kết quả    :', RESULT_DIR)
print('File tạm   :', SCRATCH)
print()
print('Giao thức  :', PROTOCOL,
      '| select_metric =', SELECT_METRIC,
      '| eval_steps =', EVAL_STEPS,
      '| seed =', SEEDS)
print('Mục tiêu   :', '88,13 (tái lập vòng 1)' if PROTOCOL == 'round1'
      else 'sàn nhiễu, không nhắm một con số cụ thể')


## 2. Dependencies

Ảnh Kaggle đã có torch, sklearn, scipy, pandas, matplotlib. Chỉ thiếu `ftfy` và `regex`
mà tokenizer của CLIP cần. Cần **Internet: On**.

In [ ]:

!pip -q install ftfy regex


## 3. Nạp Sẵn Trọng Số CLIP

`model.py` gọi `clip.load("ViT-B/16")`, vốn tải 335 MB mỗi phiên. `clip._download` kiểm
tra `~/.cache/clip/ViT-B-16.pt` bằng SHA256 trước, nên copy file vào đó là bỏ qua được
bước tải mà **không phải sửa `model.py`**.

Không có file thì cell này bỏ qua, và CLIP tự tải khi chạy.

Tải một lần bằng:
`wget https://openaipublic.azureedge.net/clip/models/5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f/ViT-B-16.pt`

In [ ]:

CLIP_SHA256 = '5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f'
cache_dir = Path.home() / '.cache' / 'clip'
cache_dir.mkdir(parents=True, exist_ok=True)
target = cache_dir / 'ViT-B-16.pt'

# walk_dirs chu khong phai glob('**'): xem ghi chu symlink o muc 1.
source = next((d / 'ViT-B-16.pt' for d in walk_dirs(INPUT_ROOT)
               if (d / 'ViT-B-16.pt').exists()), None)
if target.exists():
    print('Đã có sẵn trong cache:', target)
elif source is None:
    print('Không thấy ViT-B-16.pt trong /kaggle/input.')
    print('CLIP sẽ tự tải khi chạy (cần Internet: On).')
else:
    print('Copy', source, '->', target)
    shutil.copy2(source, target)

if target.exists():
    import hashlib
    digest = hashlib.sha256(target.read_bytes()).hexdigest()
    print('SHA256 khớp:', digest == CLIP_SHA256)
    if digest != CLIP_SHA256:
        print('  KHÔNG khớp -> clip.load sẽ tải lại. Kiểm tra lại file trong dataset.')


## 4. Preflight

Kiểm tra mọi thứ trước khi tiêu hàng giờ GPU. Bốn nhóm, và ba nhóm sau quan trọng hơn
nhóm đầu:

1. Có đủ file không.
2. **Bản code có phải bản mới không** — danh sách tham số phải có `deterministic` và
   `skip_shifted_view`. Thiếu chúng nghĩa là Dataset 1 đang giữ bản cũ, và lệnh ở mục 8
   sẽ chết ngay dòng đầu.
3. **`utils/layers.py` đã vá chưa** — bản upstream ghi cứng `.to('cuda')` trong
   `DistanceAdj.forward`. Kiểm tra bằng cách chạy thật, không phải bằng cách đọc mã.
4. Mọi file `.npy` mà hai file danh sách trỏ tới có tồn tại không.

`preflight()` được gọi tự động ở đầu mọi cell tốn thời gian, nên không thể chạy nhầm với
setup hỏng.

In [ ]:

import csv
import importlib
from collections import Counter

import numpy as np
import torch

_preflight_done = False


def preflight(force=False):
    global _preflight_done
    if _preflight_done and not force:
        return True

    problems = []

    if not torch.cuda.is_available():
        problems.append('Không có GPU. Settings -> Accelerator -> GPU T4 x2 hoặc P100.')

    if FEATURE_ROOT is None:
        print_input_tree()
        problems.append('Không dò ra dataset feature. Nhìn cây /kaggle/input ở trên, tìm thư '
                        'mục CHỨA các thư mục lớp, rồi đặt FEATURE_OVERRIDE ở mục 1. '
                        'Nếu cây trống thì dataset chưa gắn: Run -> Restart & Clear Cell Outputs.')

    need = [SRC_DIR / n for n in
            ['model.py', 'ucf_train_augment.py', 'ucf_option_augment.py',
             'ucf_shift_sensitivity.py', 'ucf_test_description.py',
             'ucf_train_class_prototype.py',
             'utils/dataset_augment.py', 'utils/tools.py', 'utils/layers.py',
             'utils/ucf_detectionMAP.py', 'clip/clip.py',
             'clip/bpe_simple_vocab_16e6.txt.gz',
             'tests/test_dataset_augment.py', 'tests/test_shift_consistency_loss.py',
             'tests/test_two_view_batching.py', 'tests/test_train_smoke.py']]
    need += [Path(TRAIN_LIST), Path(TEST_LIST), LIST_DIR / 'make_gt_ucf_relative.py',
             LIST_DIR / 'Temporal_Anomaly_Annotation.txt']
    for p in need:
        if not p.exists():
            problems.append(f'Thiếu file: {p}')

    # --- Bản code có phải bản mới không -----------------------------------------------
    if (SRC_DIR / 'ucf_option_augment.py').exists():
        import ucf_option_augment
        importlib.reload(ucf_option_augment)
        known = {action.dest for action in ucf_option_augment.parser._actions}
        needed = {'shift_ratio', 'shift_direction', 'shift_ratio_warmup', 'lambda_auto',
                  'lambda_auto_steps', 'lambda_auto_recalibrate', 'lambda_auto_max_growth',
                  'select_metric', 'run_tag', 'metrics_csv',
                  'deterministic', 'skip_shifted_view'}
        for name in sorted(needed - known):
            how = ('Code lấy từ GitHub: bạn CHƯA push bản mới lên branch '
                   f'{GITHUB_BRANCH}. Commit + push từ máy, xoá /kaggle/working/vadclip '
                   'và /kaggle/working/repo, rồi chạy lại mục 1.'
                   if CODE_FROM_GITHUB else
                   'Cập nhật Dataset code (New Version), xoá /kaggle/working/vadclip, '
                   'chạy lại mục 1.')
            problems.append(f'ucf_option_augment.py thiếu --{name.replace("_", "-")} — bản cũ. ' + how)

    # --- utils/layers.py đã vá chưa ---------------------------------------------------
    try:
        from utils.layers import DistanceAdj
        probe = DistanceAdj()                 # tham số nằm trên CPU
        if probe(2, 32).device.type != 'cpu':
            problems.append('utils/layers.py là BẢN CŨ: DistanceAdj ghi cứng .to("cuda"). '
                            'Upload lại từ repo cục bộ.')
        del probe
    except Exception as error:
        problems.append(f'Không nạp được utils/layers.py: {error!r}')

    # --- Ground truth -----------------------------------------------------------------
    global GT_MISSING
    GT_MISSING = [str(LIST_DIR / n) for n in
                  ('gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy')
                  if not (LIST_DIR / n).exists()]

    # --- Feature ----------------------------------------------------------------------
    if FEATURE_ROOT is not None:
        for name, expected in [('ucf_CLIP_rgb_relative.csv', 16100),
                               ('ucf_CLIP_rgbtest_relative.csv', 290)]:
            path = LIST_DIR / name
            if not path.exists():
                continue
            rows = list(csv.DictReader(open(path, encoding='utf-8')))
            missing = [r for r in rows if not (FEATURE_ROOT / r['path']).exists()]
            print(f'  {name}: {len(rows)} dòng (mong đợi {expected}), thiếu {len(missing)}')
            if missing:
                print('    Thiếu theo nhãn:', dict(Counter(r['label'] for r in missing)))
                for r in missing[:5]:
                    print('      ', r['path'])
                problems.append(f'{name}: thiếu {len(missing)} file feature.')

    if problems:
        print()
        print('PREFLIGHT KHÔNG ĐẠT:')
        for p in problems:
            print('  -', p)
        raise RuntimeError('Sửa các mục trên rồi chạy lại cell này.')

    print()
    print('PREFLIGHT ĐẠT')
    print('  GPU          :', torch.cuda.get_device_name(0))
    print('  torch        :', torch.__version__)
    print('  FEATURE_ROOT :', FEATURE_ROOT)
    print('  Bản code     : có --deterministic và --skip-shifted-view')
    print('  layers.py    : bản đã vá')
    free = shutil.disk_usage(WORK).free / 1e9
    print(f'  /kaggle/working còn trống: {free:.1f} GB')
    if GT_MISSING:
        print('  Thiếu ground truth (mục 5.1 sẽ sinh lại):')
        for path in GT_MISSING:
            print('     ', path)
    else:
        gt = np.load(LIST_DIR / 'gt_ucf.npy')
        print('  gt_ucf.npy   :', len(gt), 'frame |', int(gt.sum()), 'frame bất thường')

    _preflight_done = True
    return True


preflight(force=True)


## 5. Hàm Chạy Lệnh

Khác bản Colab đúng ba chỗ, và cả ba đều là hệ quả của giới hạn Kaggle:

- **Không có bước copy feature.** `/kaggle/input` đã là đĩa local, đọc thẳng. Trên Colab
  phải giải nén từ Drive vì đọc hàng nghìn file nhỏ qua Drive vừa chậm vừa hay đứt.
- **File trung gian sang `/kaggle/temp`.** Checkpoint theo epoch, `model_cur`, và
  checkpoint chọn-theo-AUC không vào Output, nên 20 GB không thành vấn đề. Đánh đổi:
  phiên kết thúc là chúng biến mất — nhưng script vốn không resume được, nên chúng không
  cứu được gì.
- **Chỉ trọng số cuối ở lại `/kaggle/working/models`.**

In [ ]:

import time


def run_command(cmd, log_name=None):
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    captured = []
    for line in process.stdout:
        print(line, end='')
        captured.append(line)
    process.wait()
    output = ''.join(captured)
    if log_name:
        (LOG_DIR / log_name).write_text(output, encoding='utf-8')
        print('Log saved:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return output


def build_train_cmd(tag, seed=234, lambda_consistency=0.0, lambda_auto=0.0,
                    recalibrate=False, max_growth=2.0,
                    shift_offset=26, shift_ratio=0.0, shift_direction='head',
                    random_shift=False, ratio_warmup=0, detach=False,
                    branch='c', warmup=1, max_epoch=None,
                    skip_shifted_view=None, deterministic=None,
                    metrics_csv=True, extra=None):
    return PY + [
        'ucf_train_augment.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--seed', seed,
        '--deterministic',
        str(DETERMINISTIC if deterministic is None else deterministic).lower(),
        '--lambda-consistency', lambda_consistency,
        '--lambda-auto', lambda_auto,
        '--lambda-auto-recalibrate', str(recalibrate).lower(),
        '--lambda-auto-max-growth', max_growth,
        '--skip-shifted-view',
        str(SKIP_SHIFTED_VIEW if skip_shifted_view is None else skip_shifted_view).lower(),
        '--shift-offset', shift_offset,
        '--shift-ratio', shift_ratio,
        '--shift-direction', shift_direction,
        '--random-shift', str(random_shift).lower(),
        '--shift-ratio-warmup', ratio_warmup,
        '--consistency-branch', branch,
        '--consistency-detach', str(detach).lower(),
        '--consistency-warmup', warmup,
        '--max-epoch', MAX_EPOCH if max_epoch is None else max_epoch,
        '--batch-size', BATCH_SIZE,
        '--lr', LR,
        '--use-pretrained-model', str(USE_PRETRAINED).lower(),
        '--pretrained-model-path', PAPER_MODEL or '',
        '--num-workers', NUM_WORKERS,
        '--pin-memory', 'true',
        '--eval-steps', EVAL_STEPS,
        '--select-metric', SELECT_METRIC,
        '--run-tag', tag,
        '--metrics-csv', METRICS_CSV if metrics_csv else '',
        # Chỉ file này vào Output. Ba file dưới là trung gian, sang /kaggle/temp.
        '--output-model-path',    MODEL_DIR / f'model_{tag}.pth',
        '--checkpoint-path',      SCRATCH / f'checkpoint_{tag}.pth',
        '--save-cur-path',        SCRATCH / f'model_cur_{tag}.pth',
        '--epoch-checkpoint-dir', SCRATCH / f'epoch_checkpoints_{tag}',
    ] + list(extra or [])


def train_run(tag, **kwargs):
    '''Bỏ qua lần chạy đã có kết quả: chạy lại cell là đi tiếp, không phải làm lại.'''
    preflight()
    output_path = MODEL_DIR / f'model_{tag}.pth'
    if output_path.exists():
        print(f'[bỏ qua] {output_path} đã tồn tại. Xoá file nếu muốn chạy lại.')
        return None
    started = time.time()
    output = run_command(build_train_cmd(tag, **kwargs), log_name=f'train_{tag}.log')
    print(f'Xong sau {(time.time() - started) / 3600:.2f} giờ')
    return output


def shift_sensitivity(tag, model_path=None, offsets=(0, 8, 16, 32)):
    '''Chỉ số CHÍNH: tương quan điểm số trước và sau khi dịch, sau khi căn chỉnh lại.'''
    preflight()
    output_dir = RESULT_DIR / f'shift_sens_{tag}'
    if (output_dir / 'shift_sensitivity_summary.csv').exists():
        print(f'[đã có] {output_dir}')
        return output_dir
    run_command(PY + [
        'ucf_shift_sensitivity.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        '--model-path', model_path or (MODEL_DIR / f'model_{tag}.pth'),
        '--gt-path', GT_ARGS[1],
        '--offsets', *offsets,
        '--output-dir', output_dir,
    ], log_name=f'sens_{tag}.log')
    return output_dir


print('Sẵn sàng. Tag của phiên này:', [f'{RUN_PREFIX}_s{s}' for s in SEEDS])


### 5.1. Sinh Lại Ground Truth (chỉ khi mục 4 báo thiếu)

Ba file `gt_*.npy` sinh từ file nhãn thời gian cộng với độ dài thật của từng file đặc
trưng, nên bắt buộc phải có feature trước.

In [ ]:

if GT_MISSING:
    run_command(PY + [
        str(LIST_DIR / 'make_gt_ucf_relative.py'),
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        # Bắt buộc: mặc định của script là 'list/Temporal_Anomaly_Annotation.txt'
        # tính theo thư mục hiện hành, mà thư mục hiện hành là src/ -> không có.
        '--annotation', str(LIST_DIR / 'Temporal_Anomaly_Annotation.txt'),
        '--output-dir', LIST_DIR,
    ], log_name='make_gt.log')
    preflight(force=True)
else:
    print('Đã có đủ ground truth, bỏ qua.')


## 6. Unit Test

Bốn file, khoảng một phút, chạy trên CPU với bộ mã hoá CLIP giả nên không cần feature
thật. Hai bài mới nằm trong `test_train_smoke.py` và chúng kiểm tra đúng thay đổi của lần
này:

- `test_skip_shifted_view_matches_the_two_view_control` chạy trọn một epoch λ = 0 theo cả
  hai đường, cùng hạt giống, rồi so ba mất mát nhiệm vụ ở bước 0 và **toàn bộ trọng số**
  sau epoch đó.
- `test_skip_shifted_view_refuses_a_live_lambda` kiểm tra script từ chối chạy khi cờ được
  bật cùng một λ khác 0 — nếu không, cờ này sẽ âm thầm vứt bỏ số hạng ràng buộc.

`test_two_view_batching.py` là bài đặt nền cho cả hai: nó chứng minh `CLIPVAD` độc lập
theo lô trên mô hình thật.

In [ ]:

for test_file in ('test_dataset_augment', 'test_shift_consistency_loss',
                  'test_two_view_batching', 'test_train_smoke'):
    run_command(PY + [f'tests/{test_file}.py'], log_name=f'{test_file}.log')


## 7. Đo Tốc Độ Và Kiểm Tra Tương Đương — CHẠY TRƯỚC MỤC 8

Hai mươi bước thật, chạy hai lần: một lần giữ view dịch, một lần bỏ. Rẻ (vài phút) và trả
lời hai câu hỏi mà không câu nào nên đoán:

**Mười epoch có kịp 12 giờ không.** Một epoch là 125 bước (8000 video Normal chia lô 64),
nên 10 epoch là 1250 bước cộng 10 lần chấm điểm trên tập test. Cell in ra ước lượng tổng.
Nếu nó vượt 10 giờ, hãy bật `SKIP_SHIFTED_VIEW = True` ở mục 1 hoặc giảm `MAX_EPOCH`
**trước khi** bắt đầu, chứ đừng phát hiện ở giờ thứ mười một.

**Bỏ view dịch có thật sự không đổi gì không.** Unit test đã kiểm tra điều này trên mô
hình tí hon với CLIP giả. Cell này kiểm tra lại trên mô hình thật, dữ liệu thật, GPU
thật: ba mất mát nhiệm vụ ở bước 0 phải trùng nhau. Đây là bằng chứng mạnh hơn unit test,
vì nó chạy đúng thứ sẽ chạy ở mục 8.

Cell không ghi trọng số nào — `--debug-max-steps` dừng vòng lặp trước mọi lệnh lưu.

In [ ]:

import re

PROBE_STEPS = 20
STEPS_PER_EPOCH = 125          # min(8000, 8100) // 64, với drop_last

step0_pattern = re.compile(
    r'\[step 0\] loss1=([\d.eE+-]+) loss2=([\d.eE+-]+) loss3=([\d.eE+-]+) '
    r'loss4_raw=([\d.eE+-]+)')

timings, losses = {}, {}
for skip in (False, True):
    label = 'bỏ view dịch' if skip else 'giữ hai view'
    print('=' * 90)
    print(f'--- {PROBE_STEPS} bước, {label} ---')
    started = time.time()
    output = run_command(
        build_train_cmd('probe', seed=SEEDS[0], skip_shifted_view=skip,
                        metrics_csv=False,
                        extra=['--debug-max-steps', str(PROBE_STEPS)]),
        log_name=f'probe_skip{int(skip)}.log')
    timings[skip] = time.time() - started
    match = step0_pattern.search(output)
    losses[skip] = [float(g) for g in match.groups()] if match else None

print()
print('=' * 90)
print('TƯƠNG ĐƯƠNG — ba mất mát nhiệm vụ ở bước 0 phải trùng nhau')
if losses[False] and losses[True]:
    worst = 0.0
    for name, a, b in zip(('loss1', 'loss2', 'loss3'), losses[False], losses[True]):
        worst = max(worst, abs(a - b))
        print(f'  {name}: hai view {a:.8f} | một view {b:.8f} | lệch {abs(a - b):.2e}')
    print(f'  loss4_raw: hai view {losses[False][3]:.8f} | một view {losses[True][3]:.8f} '
          '(một view phải bằng 0)')
    print()
    print('  => TRÙNG. Bật SKIP_SHIFTED_VIEW là an toàn.' if worst < 1e-6 else
          f'  => LỆCH {worst:.2e}. ĐỪNG bật SKIP_SHIFTED_VIEW, và báo lại chỗ này.')
else:
    print('  Không bóc được dòng [step 0] từ log — kiểm tra log ở trên.')

# --- Chi phí chấm điểm, đo thật ------------------------------------------------------
# Với --eval-steps 1280 thì mỗi epoch chấm ~12 lần, không phải 1 lần. Bỏ qua chỗ này là
# ước lượng thiếu cả tiếng đồng hồ. Chạy 11 bước với eval bật: bước thứ 10 kích hoạt
# đúng một lần chấm điểm, nên hiệu số so với chi phí thuần huấn luyện chính là nó.
per_step = timings[SKIP_SHIFTED_VIEW] / PROBE_STEPS
evals_per_epoch = 0
if EVAL_STEPS > 0:
    evals_per_epoch = (STEPS_PER_EPOCH * BATCH_SIZE * 2) // EVAL_STEPS
# train() còn chấm thêm một lần cuối mỗi epoch khi có --metrics-csv.
evals_per_epoch += 1
evals_total = evals_per_epoch * MAX_EPOCH

eval_seconds = None
if EVAL_STEPS > 0:
    print()
    print('=' * 90)
    print(f'--- đo chi phí một lần chấm điểm ({EVAL_STEPS=} -> {evals_per_epoch} lần/epoch) ---')
    started = time.time()
    run_command(build_train_cmd('probe_eval', seed=SEEDS[0], metrics_csv=False,
                                extra=['--debug-max-steps', '11']),
                log_name='probe_eval.log')
    eval_seconds = max(0.0, (time.time() - started) - per_step * 11)
    print(f'  một lần chấm điểm trên 290 video test: ~{eval_seconds:.0f} giây')

print()
print('TỐC ĐỘ — ước lượng cho', MAX_EPOCH, 'epoch')
print(f'  {STEPS_PER_EPOCH} bước/epoch · {evals_per_epoch} lần chấm/epoch · '
      f'{evals_total} lần chấm tổng cộng')
print()
for skip in (False, True):
    step_s = timings[skip] / PROBE_STEPS
    train_h = step_s * STEPS_PER_EPOCH * MAX_EPOCH / 3600
    eval_h = (eval_seconds if eval_seconds is not None else 180) * evals_total / 3600
    label = 'bỏ view dịch ' if skip else 'giữ hai view '
    flag = '  <-- cấu hình hiện tại' if skip == SKIP_SHIFTED_VIEW else ''
    print(f'  {label}: {step_s:5.2f} s/bước | huấn luyện {train_h:4.1f} giờ '
          f'+ chấm điểm {eval_h:4.1f} giờ = ~{train_h + eval_h:4.1f} giờ{flag}')
if eval_seconds is None:
    print()
    print('  (chi phí chấm điểm là con số giả định 180 giây — chỉ đo thật khi EVAL_STEPS > 0)')
print()
print('  Hạn cứng của Kaggle là 12 giờ. Trên 10 giờ là nên giảm MAX_EPOCH, tăng')
print('  EVAL_STEPS, hoặc bật SKIP_SHIFTED_VIEW — 20 bước đầu bao giờ cũng lạc quan')
print('  hơn thực tế.')
if PROTOCOL == 'round1':
    print()
    print('  Lưu ý: giao thức round1 chấm điểm ~12 lần mỗi epoch vì --select-metric auc')
    print('  cần có nhiều điểm để chọn. Đó là cái giá của việc tái lập đúng 88,13.')


## 8. Chạy Đối Chứng λ = 0

Cấu hình lấy nguyên từ mục 1, không nhận tham số riêng ở đây — để mọi lần chạy giống hệt
nhau trừ hạt giống.

```
λ = 0 · lr 2e-5 · lô 64 · 10 epoch · MultiStepLR([4,8], 0.1) · train từ đầu
--select-metric none · --eval-steps 0 · --num-workers 2
```

Cách cắt (`--shift-offset 26 --shift-direction head`) vẫn được truyền nhưng **không ảnh
hưởng gì** khi λ = 0: view dịch vẫn được tạo, đi qua mô hình, rồi bị nhân với 0. Truyền
nó để dòng lệnh khác dòng lệnh của lần chạy có ràng buộc đúng một tham số.

### Luật chọn checkpoint quyết định con số bạn nhận được

Đây là chỗ dễ mất nhiều giờ nhất nếu hiểu nhầm, nên đọc kỹ.

`--select-metric` quyết định lần chạy kết thúc bằng trọng số nào:

| Giá trị | Giữ trọng số nào | Dùng khi |
|---|---|---|
| `auc` | Tốt nhất trong các lần chấm **trên chính tập test** | Tái lập vòng 1 |
| `none` | Của epoch cuối | Đo sàn nhiễu, so hai cấu hình |

Chênh lệch giữa hai luật này **không nhỏ**, và đã đo được trên đúng cùng một hạt giống:

```text
seed 234 · lambda = 0 · lr 2e-5 · 10 epoch

vòng 1  (select-metric auc)   ->  88,13
vòng 2  (select-metric none)  ->  87,84      = v2_ctrl_s234
                                  ------
                                   0,29 điểm là của LUẬT CHỌN, không phải của mô hình
```

Nói cách khác: cùng một mô hình, cùng một quá trình huấn luyện, chỉ khác chỗ "lấy trọng
số ở đâu ra" thì con số báo cáo đã lệch 0,29 điểm. Muốn ra 88,13 thì bắt buộc dùng `auc`.

`auc` là luật của repo gốc, nhưng nó **chọn mô hình dựa trên chính tập test** — tức là số
đo được lạc quan hơn thực tế, và vòng 2 đo được rằng nó thổi độ trải giữa các lần chạy
lên khoảng một bậc độ lớn. Cặp λ = 0,01 và đối chứng của vòng 2 là ví dụ: chọn theo đỉnh
thì "thắng" 0,13 điểm, giữ trọng số cuối thì "thua" 0,15 điểm. Cùng một cặp lần chạy, đảo
dấu kết luận.

Vì vậy: dùng `round1` để **tái lập**, rồi chuyển sang `round23` khi bắt đầu **so sánh** các
cải tiến. Hai việc khác nhau, cần hai luật khác nhau.

### Ra bao nhiêu thì coi là tái lập được

Không phải đúng 88,13. Số liệu chính bạn đã đo:

| Đổi gì | AUC lệch bao nhiêu |
|---|---|
| Không đổi gì, chỉ khác epoch cuối | 0,07 điểm |
| Cùng cấu hình, cùng seed, **khác phần cứng** | 0,23 điểm |
| Khác hạt giống | 2,56 điểm |

Kaggle là phần cứng khác Colab, nên hàng thứ hai là hàng áp dụng. **Rơi vào khoảng
87,9 – 88,4 là tái lập thành công.** Ra 88,13 chằn chặn thì là may, không phải tiêu chuẩn.

Theo dõi dòng `epoch: N | loss1: ... | loss4_raw: ...`. Với λ = 0, `loss4_raw` vẫn được
tính và in ra (trừ khi bật `SKIP_SHIFTED_VIEW`) — và nó là một số liệu đáng giữ: ở vòng
2, mất mát nhất quán của lần đối chứng **tăng** đều từ 0,0037 lên 0,0121 qua mười epoch.
Huấn luyện thông thường càng lâu thì mô hình càng giòn với dịch chuyển thời gian. Đó là
động cơ của cả hướng nghiên cứu, đo trên chính lần chạy không có ràng buộc nào.

> Đừng chạy đi chạy lại rồi chọn lần đẹp nhất. Đó là chọn kết quả theo tập test ở cấp độ
> lần-chạy, và nó phá hỏng đúng cái sàn nhiễu mà lần chạy này sinh ra để đo.

In [ ]:

trained_tags = []
for seed in SEEDS:
    tag = f'{RUN_PREFIX}_s{seed}'
    print('#' * 90)
    print('ĐỐI CHỨNG λ = 0, hạt giống', seed)
    train_run(tag, seed=seed, lambda_consistency=0.0, lambda_auto=0.0,
              shift_offset=26, shift_direction='head')
    trained_tags.append(tag)

print()
print('Đã chạy:', trained_tags)


## 9. Đo Độ Nhạy Dịch Chuyển

Phép đo trả lời đúng câu hỏi mà hàm mất mát nhắm tới: dịch đầu vào đi Δ bước, căn kết quả
trở lại trục cũ, thì chuỗi điểm số có giữ nguyên không. Không huấn luyện lại gì — chỉ nạp
trọng số rồi chấm trên 290 video test, vài phút mỗi mô hình.

Đo ở Δ bằng 8, 16 và 32, cả ba đều khác 26 — độ dịch mà các lần chạy có ràng buộc được
luyện. Với đối chứng thì điều đó không quan trọng, nhưng phải đo bằng đúng thước đo ấy
thì con số mới đặt cạnh nhau được.

Checkpoint tác giả được đo kèm làm mốc: ở vòng 1, Δ = 16 cho tương quan 0,635 với đối
chứng và 0,697 với checkpoint gốc. Khoảng cách 0,062 đó là thang đo tự nhiên cho mọi
chênh lệch trong bảng này.

In [ ]:

failed = []
if PAPER_MODEL and Path(PAPER_MODEL).exists():
    try:
        shift_sensitivity('paper', model_path=PAPER_MODEL)
    except Exception as error:
        failed.append(('paper', repr(error)))
        print('[LỖI] paper:', error)
else:
    print('Không có model_ucf.pth — bỏ qua mốc đối chiếu.')

for tag in trained_tags or [f'{RUN_PREFIX}_s{s}' for s in SEEDS]:
    if not (MODEL_DIR / f'model_{tag}.pth').exists():
        print('[chưa có trọng số]', tag)
        continue
    print('=' * 90)
    try:
        shift_sensitivity(tag)
    except Exception as error:
        failed.append((tag, repr(error)))
        print(f'[LỖI] {tag}: {error}')

if failed:
    print()
    print('Thất bại:')
    for tag, error in failed:
        print(f'  {tag}: {error}')


## 10. Bảng Tổng Hợp

Ghép hai nguồn: bảng độ nhạy vừa đo, và dòng cuối của mỗi lần chạy trong CSV huấn luyện.

Lần chạy bị ngắt rồi chạy lại sẽ **nối thêm** vào cuối CSV chứ không ghi đè, nên cell bỏ
trùng theo cặp (lần chạy, epoch) và giữ dòng cuối.

Muốn đặt cạnh số liệu Colab: upload `shift_v2_metrics.csv` / `shift_v3_metrics.csv` vào
Dataset 1 rồi thêm đường dẫn vào `EXTRA_METRICS`.

In [ ]:

import pandas as pd

OFFSETS = [8, 16, 32]
NAN = float('nan')

# Thêm CSV của Colab vào đây để so trực tiếp, ví dụ: [str(CODE_ROOT / 'shift_v2_metrics.csv')]
EXTRA_METRICS = [str(f) for d in walk_dirs(INPUT_ROOT)
                 for f in sorted(d.glob('shift_v*_metrics.csv'))]


def read_sensitivity(tag):
    path = RESULT_DIR / f'shift_sens_{tag}' / 'shift_sensitivity_summary.csv'
    if not path.exists():
        return None
    frame = pd.read_csv(path).set_index('offset')
    row = {f'corr{o}': float(frame.loc[o, 'mean_classifier_corr'])
           for o in OFFSETS if o in frame.index}
    row['auc_spread'] = float(frame.iloc[0]['classifier_auc_spread_common'])
    return row


def read_train_metrics(*csv_paths):
    '''Đỉnh VÀ điểm cuối của mỗi lần chạy.

    Bản cũ chỉ lấy dòng cuối, và nó sai với giao thức round1: `--select-metric auc` lưu
    ra checkpoint TỐT NHẤT, nên con số ứng với file trọng số bạn nhận được là ĐỈNH của
    đường cong, không phải lần chấm cuối cùng. Với round1 chênh lệch này là thật:
    vòng 1 đạt đỉnh 88,13 ở epoch 6 nhưng kết thúc ở 88,07.

    Khử trùng theo (run, epoch, step) chứ không phải (run, epoch): bản cũ gộp theo epoch
    nên vứt luôn ~12 lần chấm giữa epoch, tức vứt đúng chỗ đỉnh hay nằm.
    '''
    out = {}
    for csv_path in csv_paths:
        if not csv_path or not Path(csv_path).exists():
            continue
        frame = pd.read_csv(csv_path)
        frame = frame[frame.run != 'run']                    # dòng tiêu đề bị lặp
        for column in ('epoch', 'step', 'auc', 'ap', 'lambda_used'):
            if column in frame.columns:
                frame[column] = pd.to_numeric(frame[column], errors='coerce')
        frame = frame.drop_duplicates(subset=['run', 'epoch', 'step'], keep='last')
        for tag, group in frame.groupby('run'):
            group = group.sort_values(['epoch', 'step'])
            last = group.iloc[-1]
            best = group.loc[group['auc'].idxmax()]
            out[tag] = {
                'auc_best': float(best['auc']) * 100,
                'auc_final': float(last['auc']) * 100,
                'ap_best': float(best['ap']) * 100,
                'ap_final': float(last['ap']) * 100,
                'best_epoch': int(best['epoch']),
                'best_step': int(best['step']),
                'lambda_used': float(last['lambda_used']),
                'epochs': int(last['epoch']),
                'evals': len(group),
            }
            # 'auc' phải là con số ứng với FILE TRỌNG SỐ đã lưu, không phải một con số
            # đẹp bất kỳ. select-metric auc lưu đỉnh; none lưu điểm cuối.
            keep_best = SELECT_METRIC == 'auc'
            out[tag]['auc'] = out[tag]['auc_best' if keep_best else 'auc_final']
            out[tag]['ap'] = out[tag]['ap_best' if keep_best else 'ap_final']
    return out


metrics = read_train_metrics(METRICS_CSV, *EXTRA_METRICS)
tags = ['paper'] + sorted({p.name[len('shift_sens_'):]
                           for p in RESULT_DIR.glob('shift_sens_*')} - {'paper'})

rows = []
for tag in tags:
    sens = read_sensitivity(tag)
    if sens is None:
        continue
    rows.append({'run': tag, **sens, **metrics.get(tag, {})})

if not rows:
    print('Chưa có bảng độ nhạy nào. Chạy mục 9 trước.')
else:
    table = pd.DataFrame(rows).set_index('run')
    print('=== Bảng chính ===')
    print(table.round(4).to_string())
    print()
    print(f"Cột 'auc' = con số của FILE TRỌNG SỐ đã lưu "
          f"({'đỉnh' if SELECT_METRIC == 'auc' else 'epoch cuối'}, vì --select-metric "
          f"{SELECT_METRIC}).")
    if 'auc_best' in table.columns and 'auc_final' in table.columns:
        print()
        print('=== Đỉnh so với điểm cuối ===')
        for tag in table.index:
            row = table.loc[tag]
            if pd.isna(row.get('auc_best')):
                continue
            print(f"  {tag:<20} đỉnh {row['auc_best']:.2f} "
                  f"(epoch {int(row['best_epoch'])}, step {int(row['best_step'])})"
                  f"  |  cuối {row['auc_final']:.2f}"
                  f"  |  chênh {row['auc_best'] - row['auc_final']:+.2f}"
                  f"  |  {int(row['evals'])} lần chấm")
    table.to_csv(RESULT_DIR / 'shift_kaggle_summary.csv')
    print()
    print('Saved:', RESULT_DIR / 'shift_kaggle_summary.csv')

    controls = [t for t in table.index if 'ctrl' in t]
    if len(controls) >= 2:
        print()
        print(f'=== Sàn nhiễu trên {len(controls)} lần đối chứng ===')
        for col in ('corr8', 'corr16', 'corr32', 'auc', 'ap'):
            if col in table.columns:
                values = table.loc[controls, col].dropna()
                if len(values) >= 2:
                    print(f'  {col:<12} trung bình {values.mean():8.4f}  '
                          f'độ lệch chuẩn {values.std():7.4f}  '
                          f'khoảng [{values.min():.4f}, {values.max():.4f}]')
        print()
        print('  Đây là con số quyết định: mọi hiệu ứng của λ nhỏ hơn độ lệch chuẩn này')
        print('  đều không kết luận được. Vòng 2 đo AUC 2,56 điểm trên đúng hai mẫu.')
    elif len(controls) == 1:
        print()
        print('Mới có 1 đối chứng trong phiên này. Sàn nhiễu cần ít nhất 3 lần chạy —')
        print('hai lần Colab (hạt giống 234 và 1234) cộng lần này là đủ ba, nhưng phải')
        print('gộp CSV lại: thêm đường dẫn vào EXTRA_METRICS ở đầu cell.')


## 11. Gom Sản Phẩm

Mọi thứ trong `/kaggle/working` tự thành Output của notebook. Cell này chỉ liệt kê và
cảnh báo nếu vượt hạn mức 20 GB.

Tải về bằng tab **Output** ở panel bên phải, hoặc **Save Version** để giữ vĩnh viễn.

Thứ cần mang về máy để viết báo cáo:

- `results/shift_kaggle_metrics.csv` — một dòng mỗi epoch
- `results/shift_kaggle_summary.csv` — bảng chính
- `results/shift_sens_*/` — bảng độ nhạy và hình timeline
- `results/logs/train_kg_ctrl_*.log` — nhật ký đầy đủ, có quỹ đạo `loss4_raw`
- `models/model_kg_ctrl_*.pth` — trọng số, nếu còn muốn đo thêm sau này

In [ ]:

total = 0
print('Nội dung /kaggle/working:')
for f in sorted(WORK.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        total += size
        if size > 1e6:
            print(f'  {size / 1e6:8.0f} MB  {f.relative_to(WORK)}')
print()
print(f'Tổng: {total / 1e9:.2f} GB / 20 GB')
if total > 18e9:
    print('  GẦN CHẠM HẠN MỨC. Xoá bớt file trong /kaggle/working/models.')

print()
print('File nhỏ (log, csv, png):')
for f in sorted(RESULT_DIR.rglob('*')):
    if f.is_file() and f.stat().st_size <= 1e6:
        print(f'  {f.stat().st_size / 1e3:8.0f} KB  {f.relative_to(WORK)}')


## Ghi Chú

**Khác biệt so với bản Colab.** Mục 1 (dò dataset thay vì mount Drive), không có bước
giải nén feature (`/kaggle/input` đã là đĩa local), file trung gian sang `/kaggle/temp`,
và mục 7 là bước mới. Từ mục 8 trở đi lệnh chạy giống hệt bản Colab trừ đường dẫn.

**Con số nào so với con số nào.** Lần chạy này chỉ so được với các lần chạy có ràng buộc
dùng **cùng script, cùng lịch huấn luyện, cùng luật chọn trọng số**. Nó không so được với
`baseline_ctrl` của `run_baseline_kaggle.ipynb` — đó là script khác.

**Không có resume.** `--use-pretrained-model` nạp trọng số nhưng vòng lặp vẫn chạy từ
epoch 0 với optimizer mới và scheduler mới, nên nó không phải là resume. Phiên đứt giữa
chừng là mất lần chạy.

**`SKIP_SHIFTED_VIEW` mặc định tắt là có chủ ý.** Đối chứng tồn tại để khác lần chạy có
ràng buộc đúng một biến. Mục 7 cho bạn số liệu để quyết định có nên đánh đổi điều đó lấy
thời gian hay không, thay vì phải đoán.

**Kết luận "chưa chứng minh được" vẫn là kết luận hợp lệ.** Sàn nhiễu ở mục 10 là thứ
biến nó từ một lời thú nhận thành một phát biểu có định lượng.